In [4]:
from datetime import date, timedelta
import time
import xml.etree.ElementTree as ET

import pandas as pd
import requests

In [5]:
# 'https://www.cbr.ru/scripts/XML_daily.asp?date_req=02/03/2002'
CBR_URL = 'https://www.cbr.ru/scripts/XML_daily.asp'

requests_date = date(2026, 6, 16)

params = {
    'date_req': requests_date.strftime('%d/%m/%Y')
}

cbr_api_response = requests.get(CBR_URL, params=params, timeout=30, verify=False)

print(cbr_api_response.status_code)
print(cbr_api_response.url)
print(cbr_api_response.text[:1000])

200
https://www.cbr.ru/scripts/XML_daily.asp?date_req=16%2F06%2F2026
<?xml version="1.0" encoding="windows-1251"?><ValCurs Date="16.06.2026" name="Foreign Currency Market"><Valute ID="R01010"><NumCode>036</NumCode><CharCode>AUD</CharCode><Nominal>1</Nominal><Name>Австралийский доллар</Name><Value>51,3028</Value><VunitRate>51,3028</VunitRate></Valute><Valute ID="R01020A"><NumCode>944</NumCode><CharCode>AZN</CharCode><Nominal>1</Nominal><Name>Азербайджанский манат</Name><Value>42,6184</Value><VunitRate>42,6184</VunitRate></Valute><Valute ID="R01030"><NumCode>012</NumCode><CharCode>DZD</CharCode><Nominal>100</Nominal><Name>Алжирских динаров</Name><Value>54,5049</Value><VunitRate>0,545049</VunitRate></Valute><Valute ID="R01035"><NumCode>826</NumCode><CharCode>GBP</CharCode><Nominal>1</Nominal><Name>Фунт стерлингов</Name><Value>97,1789</Value><VunitRate>97,1789</VunitRate></Valute><Valute ID="R01060"><NumCode>051</NumCode><CharCode>AMD</CharCode><Nominal>100</Nominal><Name>Армянских драмов<

C:\Users\MhW\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.cbr.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [6]:
print(cbr_api_response)

<Response [200]>


In [7]:
xml_root = ET.fromstring(cbr_api_response.text)

rate_date = pd.to_datetime(
    xml_root.attrib['Date'],
    format='%d.%m.%Y'
).date()

currency_rows = []

for currency_xml in xml_root.findall('Valute'):
    currency_rows.append({
        'rate_date': rate_date,
        'currency_code': currency_xml.findtext('CharCode'),
        'currency_name': currency_xml.findtext('Name'),
        'nominal': int(currency_xml.findtext('Nominal')),
        'rate': float(currency_xml.findtext('Value').replace(',', '.'))
    })

print(currency_rows)

currency_rates_df = pd.DataFrame(currency_rows)

currency_rates_df.head()

[{'rate_date': datetime.date(2026, 6, 16), 'currency_code': 'AUD', 'currency_name': 'Австралийский доллар', 'nominal': 1, 'rate': 51.3028}, {'rate_date': datetime.date(2026, 6, 16), 'currency_code': 'AZN', 'currency_name': 'Азербайджанский манат', 'nominal': 1, 'rate': 42.6184}, {'rate_date': datetime.date(2026, 6, 16), 'currency_code': 'DZD', 'currency_name': 'Алжирских динаров', 'nominal': 100, 'rate': 54.5049}, {'rate_date': datetime.date(2026, 6, 16), 'currency_code': 'GBP', 'currency_name': 'Фунт стерлингов', 'nominal': 1, 'rate': 97.1789}, {'rate_date': datetime.date(2026, 6, 16), 'currency_code': 'AMD', 'currency_name': 'Армянских драмов', 'nominal': 100, 'rate': 19.6846}, {'rate_date': datetime.date(2026, 6, 16), 'currency_code': 'BHD', 'currency_name': 'Бахрейнский динар', 'nominal': 1, 'rate': 192.648}, {'rate_date': datetime.date(2026, 6, 16), 'currency_code': 'BYN', 'currency_name': 'Белорусский рубль', 'nominal': 1, 'rate': 26.134}, {'rate_date': datetime.date(2026, 6, 16)

,rate_date,currency_code,currency_name,nominal,rate
0,2026-06-16,AUD,Австралийский доллар,1,51.3028
1,2026-06-16,AZN,Азербайджанский манат,1,42.6184
2,2026-06-16,DZD,Алжирских динаров,100,54.5049
3,2026-06-16,GBP,Фунт стерлингов,1,97.1789
4,2026-06-16,AMD,Армянских драмов,100,19.6846


In [8]:
cur_dzd_df = currency_rates_df.loc[currency_rates_df['currency_code'] == 'DZD', ['currency_code', 'nominal', 'rate']]
nominal_dzd = cur_dzd_df['nominal'].iloc[0]
rate_dzd = cur_dzd_df['rate'].iloc[0]
print(rate_dzd / nominal_dzd)

0.545049


In [9]:
currency_rates_df['unit_rate_check'] = (
    currency_rates_df['rate'] / currency_rates_df['nominal']
)

currency_rates_df.head()

,rate_date,currency_code,currency_name,nominal,rate,unit_rate_check
0,2026-06-16,AUD,Австралийский доллар,1,51.3028,51.302800
1,2026-06-16,AZN,Азербайджанский манат,1,42.6184,42.618400
2,2026-06-16,DZD,Алжирских динаров,100,54.5049,0.545049
3,2026-06-16,GBP,Фунт стерлингов,1,97.1789,97.178900
4,2026-06-16,AMD,Армянских драмов,100,19.6846,0.196846


In [10]:
def parse_cbr_xml_to_df(cbr_xml_text):
    xml_root = ET.fromstring(cbr_xml_text)

    rate_date = pd.to_datetime(
        xml_root.attrib['Date'],
        format='%d.%m.%Y'
    )
    
    currency_rows = []

    for currency_xml in xml_root.findall('Valute'):
        currency_rows.append({
            'rate_date': rate_date,
            'currency_code': currency_xml.findtext('CharCode'),
            'currency_name': currency_xml.findtext('Name'),
            'nominal': int(currency_xml.findtext('Nominal')),
            'rate': float(currency_xml.findtext('Value').replace(',', '.'))
        })

    currency_rates_df = pd.DataFrame(currency_rows)

    return currency_rates_df

In [11]:
currency_rates_df = parse_cbr_xml_to_df(
    cbr_api_response.text
)

currency_rates_df.head()

,rate_date,currency_code,currency_name,nominal,rate
0,2026-06-16,AUD,Австралийский доллар,1,51.3028
1,2026-06-16,AZN,Азербайджанский манат,1,42.6184
2,2026-06-16,DZD,Алжирских динаров,100,54.5049
3,2026-06-16,GBP,Фунт стерлингов,1,97.1789
4,2026-06-16,AMD,Армянских драмов,100,19.6846


In [12]:
currency_rates_df.shape
currency_rates_df.columns

Index(['rate_date', 'currency_code', 'currency_name', 'nominal', 'rate'], dtype='object')